# Transfer Bayesian Optimization：初学者向け実践ガイド

このNotebookでは、**別の実験で得た知識を新しい実験に活かす**「転移ベイズ最適化」を、小さな人工データで学びます。数式の詳しい導出は[理論ガイド](../../docs/theory/26_transfer_bayesian_optimization.md)、公開APIの整理は[利用ガイド](../../docs/guide/26_transfer_bayesian_optimization.md)を参照してください。

### このNotebookで学べること

1. **Warm-start**：過去の良好な実験条件を、新しい実験の初期候補に使う
2. **TAF**：過去モデルの予測と新しいモデルの獲得関数を組み合わせる
3. **RGPE**：過去モデルが新しい実験の順位をどれだけ説明できるかで重みを決める
4. **q>1**：複数候補を一度に提案するSequential / Jointの違い
5. **Mixed**：連続値とカテゴリ値が混在する条件の扱い

**重要：** ソースタスク（過去の実験）とターゲットタスク（今回の実験）は別物です。ソースの測定値をターゲットの測定値として使い回しません。また、候補提案関数は実験を実行しないので、実際の目的関数の評価とデータ追加は利用者が行います。

### 実行前の準備

Python環境にrobotorchanとその依存ライブラリがインストールされていることを確認してください。上から順番にセルを実行します。GPの学習・獲得関数最適化は数十秒以上かかる場合があります。ここでは説明を優先し、候補探索の試行回数を小さくしています。

In [ ]:
import torch
from botorch.models import SingleTaskGP
from robotorchan.optim.transfer.taf_sequential import propose_taf_candidate
from robotorchan.optim.transfer.rgpe_sequential import propose_rgpe_candidate

torch.set_default_dtype(torch.double)
bounds = torch.tensor([[0.0], [1.0]])
def target_objective(x):
    return -((x - 0.75) ** 2)

source_x = torch.linspace(0, 1, 6).unsqueeze(-1)
source_y = -((source_x - 0.7) ** 2)
source_model = SingleTaskGP(source_x, source_y)
target_x = torch.tensor([[0.0], [0.3], [1.0]])
target_y = target_objective(target_x)
# source_model は過去タスクのGP、target_x / target_y は新しいタスクの観測です。
print('source_x:', source_x.shape, 'source_y:', source_y.shape)
print('target_x:', target_x.shape, 'target_y:', target_y.shape)


## 1. Warm-start：過去の良好な条件から始める

まず前のセルで作った変数を確認します。

- `bounds`：探索範囲。形状は `(2, d)` で、1行目が下限、2行目が上限です。今回は `d=1` なので0〜1の範囲です。
- `source_x` / `source_y`：過去タスクの入力と評価値。今回の例では0.7付近が良好です。
- `target_x` / `target_y`：新しいタスクで**実際に評価済み**の入力と評価値。こちらは0.75付近が良好です。
- `source_model`：ソース観測から構築したGPモデル。ここでは説明用に単純な1次元GPを使います。

`select_warm_start_points` は過去の良好な入力を参考に、ターゲットで**これから評価する位置**を選びます。返り値 `initial_x` は測定結果ではありません。

次のセルでは教育用の人工目的関数 `target_objective` を直接呼んで評価します。実務ではこの部分を実験装置やシミュレーションの実行に置き換えてください。なお、このセルはWarm-startの仕組みを示す独立例であり、後続のTAFに使う `target_x` を自動更新しません。

In [ ]:
from robotorchan.optim.transfer.warm_start import select_warm_start_points

initial_x = select_warm_start_points(
    bounds, 3, source_x=source_x, source_y=source_y, seed=0,
)
initial_y = target_objective(initial_x)
print('Warm-start target observations:', initial_x, initial_y)

## 2. TAF：ソース情報を獲得関数に加える

通常のベイズ最適化では、現在のターゲットGPの**予測平均と不確実性**から、次に試す点を決めます。TAF（Transfer Acquisition Function）はそこへ、過去タスクの予測平均から得られる「良さそうな領域」の情報を加えます。

最大化では概念的に次のように表します。

\[
a_{\mathrm{TAF}}(x)=w_t\operatorname{EI}_t(x)+\sum_iw_i\max(\mu_i(x)-f_t^*,0).
\]

ここで `EI` はターゲットの期待改善量、`μ_i` はソースモデルの予測平均、`f_t^*` はターゲットの最良観測値です。

**重みの順番に注意してください。** `weights = [0.5, 0.5]` は「ソースモデルの重み0.5、ターゲットモデルの重み0.5」です。ソースが2個なら `[source1, source2, target]` の順になります。重みは非負で、合計を1にします。

次のループでは、**提案 → 人工目的関数で評価 → 観測を追加**を2回繰り返します。毎回ターゲットの観測データが増える点に注目してください。ソースとターゲットが似ていないとき、ソース重みが大き過ぎると「負の転移」が起こり得ます。

In [ ]:
weights = torch.tensor([0.5, 0.5])
for iteration in range(2):
    candidate = propose_taf_candidate(
        [source_model], target_x, target_y, bounds, weights,
        raw_samples=16, num_restarts=2, seed=iteration,
    )
    observation = target_objective(candidate)
    target_x = torch.cat([target_x, candidate])
    target_y = torch.cat([target_y, observation])
    print(f'TAF iteration={iteration + 1}, x={candidate.item():.4f}, best={target_y.max().item():.4f}')

## 3. RGPE：どの過去モデルを信頼するか学習する

RGPE（Ranking-weighted Gaussian Process Ensemble）は、ソースモデルとターゲットモデルの**候補の良し悪しの順位**を比較して重みを決めます。

1. ソースモデルで、現在のターゲット観測点を予測する
2. ターゲットモデルについては、対象点を学習から除いた **LOO（Leave-One-Out）予測**を使う
3. 実測順位と予測順位の不一致を数える
4. 観測点のペアをブートストラップ抽出し、順位の良いモデルに重みを与える
5. 重み付きのモデル予測を統合し、次の候補を提案する

LOOを使う理由は、ターゲットモデルが**学習に使用した点をそのまま予測すると有利になり過ぎる**ためです。計算には複数回のGP学習が必要です。

次のセルの戻り値は `(candidate, learned_weights)` です。TAFと違い、RGPEでは重みを自分で指定しません。重みは **ソースモデル、ターゲットモデル** の順で表示されます。

`n_bootstrap=8` は説明用の軽量設定です。実務では推定の安定性を確認し、必要に応じて回数を増やしてください。

In [ ]:
candidate, learned_weights = propose_rgpe_candidate(
    [source_model], target_x, target_y, bounds,
    n_bootstrap=8, raw_samples=16, num_restarts=2, seed=0,
)
print('RGPE candidate:', candidate)
print('Source / target weights:', learned_weights)

## 4. 結果の読み方と比較実験

これまでの例はAPIの使い方を理解するためのものです。**1回の候補提案だけでTAFとRGPEの優劣は判断できません。**

公平な比較では、同じターゲット初期観測、同じ評価予算、複数の乱数シードを用意します。横軸をターゲットでの実評価回数、縦軸をそれまでの最良観測値（best-so-far）として比較します。通常BOを基準として加え、ソースとターゲットが似ている場合だけでなく、異なる場合も試してください。

比較用のスクリプトは `examples/benchmarks/transfer_bo.py` です。RGPEはLOO再学習の分だけ計算時間が増える可能性があります。

## 5. q>1：複数の候補をまとめて提案する

`q` は**1回の呼び出しで提案する候補の個数**です。例えば `q=2` は2つの実験条件を提案します。提案しただけでは測定は完了していません。

- **Sequential greedy**：1点選び、その点でのターゲットGP予測平均を**仮の観測値（fantasy）**として条件付けて次の点を選びます。実際に測定したわけではありません。
- **Joint batch**：2点の組をまとめて最適化します。ターゲットのqEIは2点の予測の相関も考慮します。

次のセルではTAFの2方式を同じデータから比較します。 `batch_strategy="sequential"` または `"joint"` を指定します。 `mc_samples` はJointでのMonte Carloサンプル数です。

出力の形状はどちらも `(2, 1)`（2候補×1変数）です。出力値は候補の**座標**であり、目的関数の評価値ではありません。

In [ ]:
for strategy in ("sequential", "joint"):
    batch = propose_taf_candidate(
        [source_model], target_x, target_y, bounds, weights,
        q=2, batch_strategy=strategy, mc_samples=32,
        raw_samples=16, num_restarts=2, seed=0,
    )
    print(strategy, "TAF candidates:", batch)


## 6. Mixed変数：連続値とカテゴリを同時に扱う

製造条件では、例えば「温度は連続値」「材料の種類はカテゴリ」のように、変数の種類が混在します。ここでは次の2変数を考えます。

| 列番号 | 意味 | 値 |
| --- | --- | --- |
| 0 | 連続変数（例：温度を0〜1に正規化） | 0〜1 |
| 1 | カテゴリ（例：材料A/B） | 0または1 |

`mixed_x` の形状は `(4, 2)`、`mixed_y` は `(4, 1)` です。`cat_dims=[1]` は「**1列目ではなく、0始まりで数えた列番号1（2列目）**がカテゴリ」を意味します。

`fixed_features_list=[{1: 0.0}, {1: 1.0}]` は、カテゴリ列に許される値を列挙しています。0と1はカテゴリの識別子であり、**0.5という中間カテゴリは存在しません**。

この例は転移先だけの振る舞いを理解するため、ソースモデルを空リスト `[]` にしています。そのためTAFの重みはターゲット1個分の `torch.ones(1)` です。これは**Mixed最適化APIの動作確認例**であり、ソース知識を活用した転移性能の実証ではありません。RGPEでもソースが空ならターゲットだけを利用します。

`q=2` としたときの出力は `(2, 2)`（2候補×2変数）です。2列目が0か1になっているかを確認してください。Joint方式は候補の組を最適化するため、Sequentialと異なる結果になることがあります。

In [ ]:
mixed_bounds = torch.tensor([[0.0, 0.0], [1.0, 1.0]])
mixed_x = torch.tensor([[0.0, 0.0], [0.5, 1.0], [1.0, 0.0], [0.2, 1.0]])
def mixed_objective(x):
    return -((x[:, :1] - 0.7) ** 2) + 0.1 * x[:, 1:2]

mixed_y = mixed_objective(mixed_x)
categories = [{1: 0.0}, {1: 1.0}]
for strategy in ("sequential", "joint"):
    taf_batch = propose_taf_candidate(
        [], mixed_x, mixed_y, mixed_bounds, torch.ones(1),
        cat_dims=[1], fixed_features_list=categories,
        q=2, batch_strategy=strategy, mc_samples=32,
        raw_samples=16, num_restarts=2, seed=0,
    )
    print(strategy, "Mixed TAF:", taf_batch)

rgpe_batch, rgpe_weights = propose_rgpe_candidate(
    [], mixed_x, mixed_y, mixed_bounds,
    cat_dims=[1], fixed_features_list=categories,
    q=2, batch_strategy="joint", n_bootstrap=8,
    mc_samples=32, raw_samples=16, num_restarts=2, seed=0,
)
print("Mixed RGPE joint:", rgpe_batch, rgpe_weights)

print('Mixed input shape:', mixed_x.shape)
print('Allowed categories:', categories)
